# 03 — Vector knowledge base (embeddings + ChromaDB)

**Purpose:** Embed policy chunks with sentence-transformers, store in ChromaDB, persist index to `outputs/vector_index/` for RAG use. No LLM.

**Input:** `data/processed/chunked/chunks.json`

**Output:** Persisted ChromaDB collection in `outputs/vector_index/`.

In [ ]:
# Avoid TensorFlow/tensorboard loading (sentence_transformers uses PyTorch).
# If you still see "MessageFactory" or "GetPrototype" error, run in terminal:
#   pip install 'protobuf>=3.20,<4'
import os
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd() if Path.cwd().name != 'notebooks' else Path.cwd().parent
CHUNK_FILE = PROJECT_ROOT / 'data' / 'processed' / 'chunked' / 'chunks.json'
INDEX_DIR = PROJECT_ROOT / 'outputs' / 'vector_index'
INDEX_DIR.mkdir(parents=True, exist_ok=True)

if not CHUNK_FILE.exists():
    raise FileNotFoundError(f"Run notebook 02 first. Expected: {CHUNK_FILE}")

with open(CHUNK_FILE, encoding='utf-8') as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks.")

Loaded 649 chunks.


In [ ]:
from sentence_transformers import SentenceTransformer

# Model: good balance of quality and speed for regulatory text
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)
texts = [c['text'] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

2026-02-06 00:57:52.807386: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Python(33695) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
import chromadb
from chromadb.config import Settings

# Persist to disk (no server)
client = chromadb.PersistentClient(path=str(INDEX_DIR), settings=Settings(anonymized_telemetry=False))
collection_name = 'policy_chunks'
# Remove if re-running to avoid duplicate ids
try:
    client.delete_collection(collection_name)
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={'description': 'Phase 1 regulatory policy chunks'}
)

ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [
    {'source': c['source'], 'country': c['country'], 'category': c['category'], 'source_id': c.get('source_id', '')}
    for c in chunks
]
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas
)
print(f"Index saved. Collection: {collection_name}, count: {collection.count()}")

Index is persisted in `outputs/vector_index/`. Use notebook 04 to test retrieval.